# 🛡️ EdgeShield: Empirical Benchmark & Comparative Evaluation Suite
## Dual-Stream SLM vs. Traditional ML & Modern LLM Baselines on Real-World Enterprise Telemetry

**Doctoral Research Milestone: RDC-4**  
**Candidate:** Gauravkumar Raval (GLS University) | **Supervisor:** Dr. Tejaskumar Bhatt  
**Target Venues:** IEEE TDSC / IEEE TIFS / Elsevier Computers & Security  
**Execution Tag:** `BENCHMARK-KAGGLE-EXP1`  

---

### 📌 Purpose & Methodological Rigor
This standalone benchmark suite performs **100% empirical, live evaluation** of the EdgeShield framework and all comparative baselines without retraining overhead:
1. **Live Classical ML Baselines**: Evaluates Random Forest, Decision Tree, XGBoost, and LightGBM classifiers trained and tested **live on the identical split** of authentic 1.07 GB LMD-2023 telemetry.
2. **Dual-Stream SLM Architecture**: Evaluates Stream A (Log Semantic Analyzer) and Stream B (Behavioral Pattern Detector) for attack detection accuracy, Macro F1, False Positive Rate (FPR), and inference latency.
3. **Zero Hardcoded Metrics**: All precision, recall, F1, accuracy, and latency numbers are computed strictly from real predictions on held-out test records.
4. **Permanent Cloud Evidence**: Running in Kaggle provides a permanent, public, reproducible cloud execution record with complete logs and downloadable JSON benchmark artifacts.

## 1. ⚙️ Hardware & Environment Discovery
Inspect the Kaggle environment (Tesla T4 GPU, CPU cores, system RAM, and attached datasets).

In [ ]:
!nvidia-smi
import os, psutil, torch
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"[*] GPU Count: {torch.cuda.device_count()}")
ram_gb = psutil.virtual_memory().total / (1024**3)
cpu_count = os.cpu_count()
print(f"[*] System RAM: {ram_gb:.2f} GB | CPU Logical Cores: {cpu_count}")

print("\n[*] Attached Datasets in /kaggle/input:")
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    if level <= 2:
        print(f"  {root} ({len(files)} files)")

## 2. 📦 Clone Latest EdgeShield Codebase & Setup Dependencies

In [ ]:
# Clone or pull latest clean repo
import os
if not os.path.exists('/kaggle/working/slm-gpu-trainer'):
    !git clone --depth 1 -b main https://github.com/graval/slm-gpu-trainer.git /kaggle/working/slm-gpu-trainer
else:
    !cd /kaggle/working/slm-gpu-trainer && git pull origin main

# Install necessary benchmark dependencies
!pip install -q xgboost lightgbm onnxruntime scikit-learn transformers datasets

## 3. 🔍 Verify Dataset & Model Checkpoints
Resolve paths to the authentic 1.07 GB LMD-2023 dataset and model checkpoints.

In [ ]:
import os, sys
sys.path.insert(0, '/kaggle/working/slm-gpu-trainer/slm-tr')

# Recursively discover LMD-2023 CSV across /kaggle/input and /kaggle/working
lmd_path = None
for search_dir in ['/kaggle/input', '/kaggle/working']:
    for root, dirs, files in os.walk(search_dir):
        if 'lmd_2023_dataset.csv' in files:
            lmd_path = os.path.join(root, 'lmd_2023_dataset.csv')
            break
    if lmd_path:
        break

print(f"[+] LMD-2023 Telemetry Dataset Path: {lmd_path}")
if lmd_path and os.path.exists(lmd_path):
    size_mb = os.path.getsize(lmd_path) / (1024**2)
    print(f"    File Size: {size_mb:.2f} MB ({size_mb/1024:.2f} GB)")

# Recursively discover model checkpoints directory
models_root = None
for search_dir in ['/kaggle/input', '/kaggle/working']:
    for root, dirs, files in os.walk(search_dir):
        if 'deberta-lateral-movement-v2_sliding_window' in dirs:
            models_root = root
            break
    if models_root:
        break

if not models_root:
    models_root = '/kaggle/working/slm-gpu-trainer/slm-tr/models_v4/models'
print(f"[+] Model Weights Root Directory: {models_root}")
if os.path.exists(models_root):
    print("    Available Models:", os.listdir(models_root))


## 4. 🚀 Execute Empirical Benchmark Suite (100% Live)
Runs the purified benchmark script `scripts/benchmark_edgeshield.py`:
- 2,000 samples for Stream A (LSA) and Stream B (BPD)
- Live training & evaluation of Random Forest, Decision Tree, XGBoost, and LightGBM
- Dual-Stream fusion and Multi-Stage attack scenario verification

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '/kaggle/working/slm-gpu-trainer/slm-tr/scripts/benchmark_edgeshield.py',
    '--samples', '2000',
    '--lmd_path', lmd_path if lmd_path else 'data/lmd_2023_dataset.csv',
    '--optc_path', '/kaggle/working/slm-gpu-trainer/slm-tr/data/optc_test_benchmark.csv',
    '--ransomware_path', '/kaggle/working/slm-gpu-trainer/slm-tr/data/ransomware_behavioral_traces.csv',
    '--export_json', '/kaggle/working/benchmark_results.json'
]

# If models exist in models_root, specify them
deberta_cand = os.path.join(models_root, 'deberta-lateral-movement-v2_sliding_window')
bpd_cand = os.path.join(models_root, 'edgeshield_bpd')
if os.path.exists(deberta_cand):
    cmd.extend(['--lsa_model', deberta_cand])
if os.path.exists(bpd_cand):
    cmd.extend(['--bpd_model', bpd_cand])

print("[*] Executing Command:", " ".join(cmd))
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd='/kaggle/working/slm-gpu-trainer/slm-tr')
for line in p.stdout:
    print(line, end='', flush=True)
p.wait()
print(f"\n[+] Process exited with return code: {p.returncode}")

## 5. ⚡ ONNX INT8 Quantization & Latency Benchmark
Benchmark runtime inference latency and memory working set on CPU and ONNX INT8 runtime.

In [ ]:
!python /kaggle/working/slm-gpu-trainer/slm-tr/scripts/benchmark_onnx.py --runs 200


## 6. 📊 Comparative Matrix & Exported Findings
Display the verified empirical metrics and save artifacts for permanent reference.

In [ ]:
import json, os
import pandas as pd

results_path = '/kaggle/working/benchmark_results.json'
if not os.path.exists(results_path):
    results_path = '/kaggle/working/slm-gpu-trainer/slm-tr/benchmark_results.json'

if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        data = json.load(f)
    
    print("=" * 80)
    print("             [EMPIRICAL MODEL COMPARISON: TABLE III]")
    print("=" * 80)
    df_models = pd.DataFrame(data.get('model_head_to_head', []))
    display(df_models[['model', 'params', 'accuracy', 'f1_macro', 'latency_cpu_ms', 'latency_onnx_int8_ms', 'ram_display', 'source']])
    
    print("\n" + "=" * 80)
    print("             [CLASSICAL & ARCHITECTURAL BASELINES]")
    print("=" * 80)
    df_baselines = pd.DataFrame(data.get('baseline_comparison', []))
    display(df_baselines[['framework', 'f1_macro', 'precision', 'recall', 'latency_ms', 'source']])
    
    print("\n[+] Benchmark Suite successfully completed. Results persisted to:", results_path)
else:
    print("[!] benchmark_results.json not found.")